In [35]:
from enum import Enum
import random
from collections import defaultdict

In [36]:
# ──────────────────────────────────────────────
# Actions
# ──────────────────────────────────────────────
 
class Action(Enum):
    COOPERATE = "C"
    DEFECT    = "D"

In [37]:
# ──────────────────────────────────────────────
# Default payoff matrix  (row player's payoff)
#
#                Opponent: C    Opponent: D
#  Player: C        3              0
#  Player: D        5              1
# ──────────────────────────────────────────────
 
DEFAULT_PAYOFFS = {
    (Action.COOPERATE, Action.COOPERATE): (3, 3),
    (Action.COOPERATE, Action.DEFECT):    (0, 5),
    (Action.DEFECT,    Action.COOPERATE): (5, 0),
    (Action.DEFECT,    Action.DEFECT):    (1, 1),
}
 

In [38]:
# ──────────────────────────────────────────────
# Built-in strategies
# ──────────────────────────────────────────────
 

def always_cooperate(my_history, opponent_history):
    return Action.COOPERATE
 
 
def always_defect(my_history, opponent_history):
    return Action.DEFECT
 
 
def tit_for_tat(my_history, opponent_history):
    """Cooperate on round 1; then copy opponent's last move."""
    if not opponent_history:
        return Action.COOPERATE
    return opponent_history[-1]
 
 
def random_strategy(my_history, opponent_history):
    return random.choice(list(Action))
 
 
def grudger(my_history, opponent_history):
    """Cooperate until the opponent defects once, then always defect."""
    if Action.DEFECT in opponent_history:
        return Action.DEFECT
    return Action.COOPERATE
 
 
def tit_for_two_tats(my_history, opponent_history):
    """Defect only if opponent defected in both of the last two rounds."""
    if len(opponent_history) >= 2 and opponent_history[-1] == Action.DEFECT \
            and opponent_history[-2] == Action.DEFECT:
        return Action.DEFECT
    return Action.COOPERATE
 
 
def two_tits_for_tat(my_history, opponent_history):
    """Retaliate with TWO defections for every one defection by opponent."""
    if len(my_history) == 0:
        return Action.COOPERATE
    # If opponent defected last round, defect
    if opponent_history and opponent_history[-1] == Action.DEFECT:
        return Action.DEFECT
    # Also defect if we were retaliating (opponent defected 2 rounds ago)
    if len(opponent_history) >= 2 and opponent_history[-2] == Action.DEFECT:
        return Action.DEFECT
    return Action.COOPERATE
 
 
def generous_tit_for_tat(my_history, opponent_history, forgiveness=0.1):
    """
    Like TitForTat but occasionally forgives a defection with probability
    `forgiveness`. Helps escape mutual defection spirals.
    """
    if not opponent_history:
        return Action.COOPERATE
    if opponent_history[-1] == Action.DEFECT:
        if random.random() < forgiveness:
            return Action.COOPERATE   # forgive
        return Action.DEFECT
    return Action.COOPERATE
 
 
def pavlov(my_history, opponent_history):
    """
    Win-Stay, Lose-Shift: repeat last move if it scored well (both C or
    player D vs opponent C), otherwise switch.
    """
    if not my_history:
        return Action.COOPERATE
    last_mine = my_history[-1]
    last_opp  = opponent_history[-1] if opponent_history else Action.COOPERATE
    # "Win" situations: (C,C) or (D,C)
    if (last_mine == Action.COOPERATE and last_opp == Action.COOPERATE) or \
       (last_mine == Action.DEFECT    and last_opp == Action.COOPERATE):
        return last_mine          # stay
    # "Lose" situations: (C,D) or (D,D)
    return Action.DEFECT if last_mine == Action.COOPERATE else Action.COOPERATE
 
 
def soft_majority(my_history, opponent_history):
    """
    Cooperate as long as the opponent has cooperated at least half the time;
    otherwise defect. Starts cooperative on ties.
    """
    if not opponent_history:
        return Action.COOPERATE
    coop_count = opponent_history.count(Action.COOPERATE)
    if coop_count >= len(opponent_history) / 2:
        return Action.COOPERATE
    return Action.DEFECT
 
 
def hard_majority(my_history, opponent_history):
    """
    Defect unless the opponent has cooperated strictly MORE than half the time.
    More suspicious than soft_majority.
    """
    if not opponent_history:
        return Action.DEFECT
    coop_count = opponent_history.count(Action.COOPERATE)
    if coop_count > len(opponent_history) / 2:
        return Action.COOPERATE
    return Action.DEFECT
 
 
def prober(my_history, opponent_history):
    """
    Play D, C, C in the first 3 rounds to probe the opponent.
    If opponent never retaliated (all C), always defect.
    Otherwise fall back to TitForTat.
    """
    round_num = len(my_history)
    if round_num == 0:
        return Action.DEFECT
    if round_num == 1:
        return Action.COOPERATE
    if round_num == 2:
        return Action.COOPERATE
    # After probe: if opponent cooperated through all 3 probe rounds, exploit
    if all(a == Action.COOPERATE for a in opponent_history[:3]):
        return Action.DEFECT
    # Otherwise TitForTat
    return opponent_history[-1]
 
 
def gradual(my_history, opponent_history):
    """
    Punishes each defection with an escalating run of defections equal to the
    total number of times the opponent has defected so far, then calms down
    with 2 cooperations.
    """
    if not opponent_history:
        return Action.COOPERATE
 
    defect_count = opponent_history.count(Action.DEFECT)
    round_num    = len(my_history)
 
    # Replay the punishment schedule up to this round
    punish_end = 0
    calm_end   = 0
    for i, move in enumerate(opponent_history):
        if move == Action.DEFECT:
            total_d_so_far = opponent_history[:i + 1].count(Action.DEFECT)
            punish_end = calm_end + total_d_so_far
            calm_end   = punish_end + 2
 
    if round_num < punish_end:
        return Action.DEFECT
    if round_num < calm_end:
        return Action.COOPERATE
    return Action.COOPERATE
 
 
def remorseful_prober(my_history, opponent_history, probe_prob=0.05):
    """
    Mostly TitForTat, but occasionally probes with a random defection.
    If the probe caused the opponent to defect, cooperates next round
    (shows remorse) to avoid a retaliation spiral.
    """
    if not opponent_history:
        return Action.COOPERATE
 
    # Detect if our last move was an unprovoked defection (probe)
    if len(my_history) >= 2:
        last_mine = my_history[-1]
        prev_opp  = opponent_history[-2] if len(opponent_history) >= 2 else Action.COOPERATE
        # We defected but opponent hadn't defected before → it was a probe
        was_probe = (last_mine == Action.DEFECT and prev_opp == Action.COOPERATE)
        if was_probe and opponent_history[-1] == Action.DEFECT:
            return Action.COOPERATE   # remorse: forgive the retaliation
 
    # Random probe
    if random.random() < probe_prob:
        return Action.DEFECT
 
    # Default: TitForTat
    return opponent_history[-1]
 

In [39]:
# ──────────────────────────────────────────────
# Player
# ──────────────────────────────────────────────

class Player:
    """
    Represents a player in an iterated Prisoner's Dilemma game.

    Parameters
    ----------
    name     : display name
    strategy : callable(my_history, opponent_history) -> Action
    """

    def __init__(self, name: str, strategy=tit_for_tat):
        self.name     = name
        self.strategy = strategy
        self.history  : list[Action] = []   # moves this player made
        self.score    : int          = 0

    # ── core ──────────────────────────────────

    def choose_action(self, opponent_history: list[Action]) -> Action:
        """Ask the strategy for the next move and record it."""
        action = self.strategy(self.history, opponent_history)
        self.history.append(action)
        return action

    def add_score(self, points: int):
        self.score += points

    # ── helpers ───────────────────────────────

    def reset(self):
        """Clear history and score (use between independent experiments)."""
        self.history = []
        self.score   = 0

    def cooperation_rate(self) -> float:
        if not self.history:
            return 0.0
        return self.history.count(Action.COOPERATE) / len(self.history)

    def __repr__(self):
        return (f"Player(name={self.name!r}, "
                f"score={self.score}, "
                f"coop_rate={self.cooperation_rate():.0%})")

In [40]:
class Game:
    """
    Runs Prisoner's Dilemma experiments between Player objects.

    Parameters
    ----------
    payoffs : dict mapping (Action, Action) -> (int, int)
              Defaults to the classic PD payoff matrix.
    """

    def __init__(self, payoffs: dict = None):
        self.payoffs = payoffs or DEFAULT_PAYOFFS

    # ── single match ──────────────────────────

    def play_match(self, player1: Player, player2: Player,
                   rounds: int = 10, reset: bool = True,
                   verbose: bool = False) -> dict:
        """
        Play one iterated match between two players.

        Returns a result dict with scores and histories.
        """
        if reset:
            player1.reset()
            player2.reset()

        p2_history: list[Action] = []   # separate from player2.history so
        p1_history: list[Action] = []   # each player sees the right view

        for r in range(1, rounds + 1):
            a1 = player1.strategy(p1_history, p2_history)
            a2 = player2.strategy(p2_history, p1_history)

            p1_history.append(a1)
            p2_history.append(a2)
            player1.history.append(a1)
            player2.history.append(a2)

            s1, s2 = self.payoffs[(a1, a2)]
            player1.add_score(s1)
            player2.add_score(s2)

            if verbose:
                print(f"  Round {r:>3}: "
                      f"{player1.name}={a1.value}({s1})  "
                      f"{player2.name}={a2.value}({s2})")

        return {
            "player1": player1.name,
            "player2": player2.name,
            "score1":  player1.score,
            "score2":  player2.score,
            "winner":  self._winner(player1, player2),
        }

    # ── tournament ────────────────────────────

    def run_tournament(self, players: list[Player],
                       rounds_per_match: int = 10,
                       verbose: bool = False) -> list[dict]:
        """
        Round-robin tournament: every pair plays one match.

        Returns a leaderboard (list of dicts) sorted by total score.
        """
        total_scores = defaultdict(int)
        results      = []

        pairs = [(players[i], players[j])
                 for i in range(len(players))
                 for j in range(i + 1, len(players))]

        for p1, p2 in pairs:
            if verbose:
                print(f"\n{'─'*40}")
                print(f"  {p1.name} vs {p2.name}")
                print(f"{'─'*40}")

            result = self.play_match(p1, p2, rounds=rounds_per_match,
                                     reset=True, verbose=verbose)
            total_scores[p1.name] += result["score1"]
            total_scores[p2.name] += result["score2"]
            results.append(result)

        leaderboard = sorted(
            [{"player": name, "total_score": score}
             for name, score in total_scores.items()],
            key=lambda x: x["total_score"],
            reverse=True,
        )
        return leaderboard

    # ── repeated experiment ───────────────────

    def run_experiment(self, player1: Player, player2: Player,
                       rounds: int = 10, repetitions: int = 100) -> dict:
        """
        Repeat the same match many times and aggregate statistics.
        Useful for strategies with randomness.
        """
        scores1, scores2 = [], []

        for _ in range(repetitions):
            result = self.play_match(player1, player2,
                                     rounds=rounds, reset=True)
            scores1.append(result["score1"])
            scores2.append(result["score2"])

        def stats(values):
            avg = sum(values) / len(values)
            return {"avg": round(avg, 2),
                    "min": min(values),
                    "max": max(values)}

        return {
            "player1": player1.name,
            "player2": player2.name,
            "repetitions": repetitions,
            "rounds_per_match": rounds,
            "stats1": stats(scores1),
            "stats2": stats(scores2),
        }

    # ── helpers ───────────────────────────────

    @staticmethod
    def _winner(p1: Player, p2: Player) -> str:
        if p1.score > p2.score:
            return p1.name
        if p2.score > p1.score:
            return p2.name
        return "Draw"

    def print_payoff_matrix(self):
        print("\nPayoff Matrix (player1_payoff, player2_payoff)")
        print(f"{'':>12} | {'C':>8} | {'D':>8}")
        print("-" * 33)
        for a1 in Action:
            row = f"{a1.value:>12} |"
            for a2 in Action:
                s1, s2 = self.payoffs[(a1, a2)]
                row += f"  ({s1},{s2})  |"
            print(row)


In [47]:
 
if __name__ == "__main__":
    game = Game()
    game.print_payoff_matrix()
 
    # --- Single match ---
    print("\n\n=== Single Match: TitForTat vs AlwaysDefect ===")
    tft    = Player("TitForTat",      tit_for_tat)
    defector = Player("AlwaysDefect", always_defect)
    result = game.play_match(tft, defector, rounds=5, verbose=True)
    print(f"\nResult: {result}")
 
    # --- Tournament ---
    print("\n\n=== Round-Robin Tournament (14 strategies) ===")
    players = [
        Player("AlwaysCooperate",   always_cooperate),
        Player("AlwaysDefect",      always_defect),
        Player("TitForTat",         tit_for_tat),
        Player("Grudger",           grudger),
        Player("TitFor2Tats",       tit_for_two_tats),
        Player("Random",            random_strategy),
        Player("2TitsForTat",       two_tits_for_tat),
        Player("GenerousTFT",       generous_tit_for_tat),
        Player("Pavlov",            pavlov),
        Player("SoftMajority",      soft_majority),
        Player("HardMajority",      hard_majority),
        Player("Prober",            prober),
        Player("Gradual",           gradual),
        Player("RemorsefulProber",  remorseful_prober),
    ]
    leaderboard = game.run_tournament(players, rounds_per_match=20)
    print(f"\n{'Rank':<6}{'Player':<20}{'Total Score':>12}")
    print("─" * 38)
    for rank, entry in enumerate(leaderboard, 1):
        print(f"{rank:<6}{entry['player']:<20}{entry['total_score']:>12}")
 
    # --- Repeated experiment ---
    print("\n\n=== Repeated Experiment: TitForTat vs Random (100 runs) ===")
    tft2   = Player("TitForTat", tit_for_tat)
    rand   = Player("Random",    random_strategy)
    exp    = game.run_experiment(tft2, rand, rounds=20, repetitions=100)
    print(f"  {exp['player1']} avg score: {exp['stats1']['avg']}")
    print(f"  {exp['player2']} avg score: {exp['stats2']['avg']}")
 


Payoff Matrix (player1_payoff, player2_payoff)
             |        C |        D
---------------------------------
           C |  (3,3)  |  (0,5)  |
           D |  (5,0)  |  (1,1)  |


=== Single Match: TitForTat vs AlwaysDefect ===
  Round   1: TitForTat=C(0)  AlwaysDefect=D(5)
  Round   2: TitForTat=D(1)  AlwaysDefect=D(1)
  Round   3: TitForTat=D(1)  AlwaysDefect=D(1)
  Round   4: TitForTat=D(1)  AlwaysDefect=D(1)
  Round   5: TitForTat=D(1)  AlwaysDefect=D(1)

Result: {'player1': 'TitForTat', 'player2': 'AlwaysDefect', 'score1': 4, 'score2': 9, 'winner': 'AlwaysDefect'}


=== Round-Robin Tournament (14 strategies) ===

Rank  Player               Total Score
──────────────────────────────────────
1     GenerousTFT                  711
2     SoftMajority                 703
3     TitForTat                    698
4     Gradual                      687
5     RemorsefulProber             682
6     TitFor2Tats                  670
7     Grudger                      653
8     Pavlov  